In [3]:
import duckdb as db

In [4]:
def read_parquet(resourse):
    try:
        path = f"../SILVER/{resourse}.parquet"
        df = db.read_parquet(path).to_df()
        print(f"Successfully read {resourse} ")
        return df
    except Exception as e:
        print(f"Error reading {resourse}: {e}")
        return None

In [5]:
launches = read_parquet("launches")
launchpads = read_parquet("launchpads")
rockets = read_parquet("rockets")

Successfully read launches 
Successfully read launchpads 
Successfully read rockets 


# launch_summary_by_year 

In [13]:
gold_launch_summary_by_year = db.sql("""
                                        select launch_year , total_launches , successful_launches , failed_launches , upcoming_launches ,
                                        round((successful_launches*100.0/total_launches),2) as success_rate_pct
                                        from (select launch_year ,
                                        count(*) as total_launches ,
                                        (select count(*) from launches l2 where l2.launch_status = 'Success'  and l2.launch_year = l.launch_year)  as successful_launches ,
                                        (select count(*) from launches l2 where l2.launch_status = 'Failed'  and l2.launch_year = l.launch_year)  as failed_launches ,
                                        (select count(*) from launches l2 where l2.launch_status = 'Upcoming'  and l2.launch_year = l.launch_year)  as upcoming_launches ,
                                        from launches l
                                        group by launch_year
                                        
                                        ) t
                                        order by launch_year 
                                    """ )
gold_launch_summary_by_year.show()

┌─────────────┬────────────────┬─────────────────────┬─────────────────┬───────────────────┬──────────────────┐
│ launch_year │ total_launches │ successful_launches │ failed_launches │ upcoming_launches │ success_rate_pct │
│    int32    │     int64      │        int64        │      int64      │       int64       │      double      │
├─────────────┼────────────────┼─────────────────────┼─────────────────┼───────────────────┼──────────────────┤
│        2006 │              1 │                   0 │               1 │                 0 │              0.0 │
│        2007 │              1 │                   0 │               1 │                 0 │              0.0 │
│        2008 │              2 │                   1 │               1 │                 0 │             50.0 │
│        2009 │              1 │                   1 │               0 │                 0 │            100.0 │
│        2010 │              2 │                   2 │               0 │                 0 │            

#  rocket_usage

In [23]:
gold_rocket_usage = db.sql("""
                              select rocket_name , type , total_launches , successful_launches ,
                              round((successful_launches * 100.0 / total_launches),2) as success_rate_pct ,
                              round((total_launches * 1.0 / active_years),2) as avg_launches_per_year
                              from ( select r.rocket_name , r.type ,
                              count(*) as total_launches ,
                              (select count(*) from launches l2 where l2.rocket_id = l.rocket_id and l2.launch_status = 'Success' ) as successful_launches ,
                              count(distinct l.launch_year) as active_years     
                              from launches l
                              join rockets r on l.rocket_id = r.rocket_id   
                              group by r.rocket_name , r.type , l.rocket_id
                              ) t    
                            order by total_launches desc
                            """)

gold_rocket_usage.show()

┌──────────────┬─────────┬────────────────┬─────────────────────┬──────────────────┬───────────────────────┐
│ rocket_name  │  type   │ total_launches │ successful_launches │ success_rate_pct │ avg_launches_per_year │
│   varchar    │ varchar │     int64      │        int64        │      double      │        double         │
├──────────────┼─────────┼────────────────┼─────────────────────┼──────────────────┼───────────────────────┤
│ Falcon 9     │ rocket  │            195 │                 176 │            90.26 │                 16.25 │
│ Falcon Heavy │ rocket  │              5 │                   3 │             60.0 │                  1.67 │
│ Falcon 1     │ rocket  │              5 │                   2 │             40.0 │                  1.25 │
└──────────────┴─────────┴────────────────┴─────────────────────┴──────────────────┴───────────────────────┘



# launchpad_activity

In [19]:
gold_launchpad_activity = db.sql("""
                                select launchpad_name , region , launch_attempts as total_launches , launch_successes as successful_launches ,
                                 round((launch_successes * 100.0) / NULLIF(launch_attempts, 0), 2) as success_rate_pct
                                 
                                from launchpads
                                
                                 """)
gold_launchpad_activity.show()

┌─────────────────┬──────────────────┬────────────────┬─────────────────────┬──────────────────┐
│ launchpad_name  │      region      │ total_launches │ successful_launches │ success_rate_pct │
│     varchar     │     varchar      │     int64      │        int64        │      double      │
├─────────────────┼──────────────────┼────────────────┼─────────────────────┼──────────────────┤
│ VAFB SLC 3W     │ California       │              0 │                   0 │             NULL │
│ CCSFS SLC 40    │ Florida          │             99 │                  97 │            97.98 │
│ STLS            │ Texas            │              0 │                   0 │             NULL │
│ Kwajalein Atoll │ Marshall Islands │              5 │                   2 │             40.0 │
│ VAFB SLC 4E     │ California       │             28 │                  27 │            96.43 │
│ KSC LC 39A      │ Florida          │             55 │                  55 │            100.0 │
└─────────────────┴───────────

# recent_mission_timeline

In [20]:
gold_recent_mission_timeline = db.sql("""
                                select l.launch_date , l.mission_name , r.rocket_name , lp.launchpad_name , l.launch_status , l.launch_status as status
                                from launches l
                                join rockets r on l.rocket_id = r.rocket_id
                                join launchpads lp on l.launchpad_id = lp.launchpad_id
                                order by l.launch_date desc
                                      """)
gold_recent_mission_timeline.show()


┌─────────────────────┬────────────────────────────┬──────────────┬─────────────────┬───────────────┬──────────┐
│     launch_date     │        mission_name        │ rocket_name  │ launchpad_name  │ launch_status │  status  │
│      timestamp      │          varchar           │   varchar    │     varchar     │    varchar    │ varchar  │
├─────────────────────┼────────────────────────────┼──────────────┼─────────────────┼───────────────┼──────────┤
│ 2022-12-05 00:00:00 │ SWOT                       │ Falcon 9     │ VAFB SLC 4E     │ Upcoming      │ Upcoming │
│ 2022-12-01 00:00:00 │ O3b mPower 3.4             │ Falcon 9     │ CCSFS SLC 40    │ Upcoming      │ Upcoming │
│ 2022-12-01 00:00:00 │ Viasat-3 & Arcturus        │ Falcon Heavy │ KSC LC 39A      │ Upcoming      │ Upcoming │
│ 2022-12-01 00:00:00 │ WorldView Legion 1 & 2     │ Falcon 9     │ CCSFS SLC 40    │ Upcoming      │ Upcoming │
│ 2022-12-01 00:00:00 │ TTL-1                      │ Falcon 9     │ VAFB SLC 4E     │ Upcoming  

# year_month_trend

In [16]:
gold_year_month_trend = db.sql("""
                                select launch_year , launch_month , count(*) as total_launches ,
                               from launches
                               group by launch_year , launch_month
                               order by launch_year , launch_month
                               """)
gold_year_month_trend.show()

┌─────────────┬──────────────┬────────────────┐
│ launch_year │ launch_month │ total_launches │
│    int32    │    int32     │     int64      │
├─────────────┼──────────────┼────────────────┤
│        2006 │            3 │              1 │
│        2007 │            3 │              1 │
│        2008 │            8 │              1 │
│        2008 │            9 │              1 │
│        2009 │            7 │              1 │
│        2010 │            6 │              1 │
│        2010 │           12 │              1 │
│        2012 │            5 │              1 │
│        2012 │           10 │              1 │
│        2013 │            3 │              1 │
│          ·  │            · │              · │
│          ·  │            · │              · │
│          ·  │            · │              · │
│        2022 │            3 │              3 │
│        2022 │            4 │              6 │
│        2022 │            5 │              5 │
│        2022 │            6 │          